In [14]:
import nltk
import pickle
import string
import pandas as pd
import os

from nltk.probability import FreqDist
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.classify import NaiveBayesClassifier, accuracy

nltk.download(['punkt', 'stopwords', 'averaged_perceptron_tagger', 'wordnet', 'omw-1.4'])

MODEL_PATH = "./model.pickle"
CSV_PATH = "./financial_dataset.csv"

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dianr\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dianr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\dianr\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dianr\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\dianr\AppData\Roaming\nltk_data...


In [15]:
df = pd.read_csv(CSV_PATH)
df.head()

,Statement,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,$SPY wouldn't be surprised to see a green close,positive
4,Shell's $70 Billion BG Deal Meets Shareholder ...,negative


In [16]:
lemmatizer = WordNetLemmatizer()

In [17]:
# n v a r
# n = noun, v = verb, a = adjective, r = adverb 
def get_label(tag):
    if tag.startswith('j'):
        return 'a'
    elif tag.startswith('r') or tag.startswith('v') or tag.startswith('n'):
        return tag [0]
    else:
        return 'n'

def lemmatizing(word_list):
    lemma_list = []
    tagged = pos_tag(word_list)
    for word, tag in tagged:
        label = get_label(tag.lower())
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemma_list.append(result)
        else:
            result = lemmatizer.lemmatize(word)
            lemma_list.append(result)
            
    return lemma_list

def preprocessing(sentence):
    eng_stopwords = stopwords.words('english')
    punctuation = string.punctuation

    word_list = word_tokenize(sentence.lower())
    removed_stopwords = [token for token in word_list if token not in eng_stopwords]
    removed_punctuation = [token for token in removed_stopwords if token not in punctuation]
    lemma_list = lemmatizing(removed_punctuation)

    return lemma_list

In [18]:
sentence = "Hello everyone, today's classes is NLP"

preprocessing(sentence)

LookupError: 
**********************************************************************
  Resource [93maveraged_perceptron_tagger_eng[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('averaged_perceptron_tagger_eng')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtaggers/averaged_perceptron_tagger_eng/[0m

  Searched in:
    - 'C:\\Users\\dianr/nltk_data'
    - 'c:\\Anaconda\\nltk_data'
    - 'c:\\Anaconda\\share\\nltk_data'
    - 'c:\\Anaconda\\lib\\nltk_data'
    - 'C:\\Users\\dianr\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [ ]:
X = df['Statement']
y = df['Sentiment']

all_sentence = " ".join(X)
all_token = preprocessing(all_sentence)

freq_dist = FreqDist(all_token)

print(freq_dist.most_common(10))

In [ ]:
def extract_features(statement):
    features = {}
    for word in freq_dist.keys():
        features[word] = (word in statement)
    
    return features

In [ ]:
feature_sets = []

# [({word : boolean}, statement)]

for(statement, sentiment) in zip(X, y):
    features = extract_features(preprocessing(statement))
    feature_sets.append((features, sentiment))

print(feature_sets[0]) 

In [ ]:
statement = ""
classifier = None

while true:
    print("1. Write Your Statement")
    print("2. Analyze Your Statament")
    print("3. Exit")

    choice = input(">> ")

    if(choice == "1"):
        # Function 1
        statement = write_statement